# Figure S2B, S2C & S2D: Hierarchical RL Modeling (Online Prolific Cohort)

| Field | Value |
|---|---|
| **Manuscript panels / stats** | Fig S2B (integrated BIC across 6 models, n=200), Fig S2C (RW alpha/beta joint density, n=200), Fig S2D (Day-1 binomial glmer fixed effects with age & sex, n=193; RPE weight=0.12, p=0.01) |
| **Inputs** | `$MEMORYBANDIT_DATA_DIR/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{learn_df_prolific_v2.csv,full_data_df_prolific_v2.csv,demographics.csv}` |
| **Outputs** | `results/hierarchical_pyem/prolific_v1/{model_comparison_delta_bic.pdf,rw_alpha_beta_joint.pdf,glmer_hit_coefficients_day_1.pdf,combined_df_lme4_hit_rpe_new_day_1.csv}` |
| **Dependencies** | `membandit`, `rlmodels_k99` (vendored), `pyem` (pinned commit `c0d09807`), R `lme4` (`R/glmer_recognition_online.R`) |
| **Expected runtime** | ~30-60 minutes for 200-subject 6-model EM fits (instant from saved CSVs) |
| **Upstream steps** | `00_prolific_preprocessing.ipynb`; run `Rscript R/glmer_recognition_online.R` before the Fig S2D plot cell |


# Hierarchical modeling — Prolific delayed-memory cohort

Replication of the six-model pyEM fitting, integrated-BIC comparison, and publication plots on the **non-EMU Prolific** delayed-memory-test sample.

**Final manuscript models:** `wsls`, `rw`, `rwf`, `rw_asym`, `bayes_fixed_reward`, `vkf_omega`.

**Data:** `learn_df_prolific_v2.csv` / `full_data_df_prolific_v2.csv` from `prolific_preprocessing.ipynb` (good-DM flag from learning rewards only: **200** Day-1 / **161** Day-2 subjects after `good_dm_performance` filter; `day_1` immediate + `day_2` 24 h delayed recognition).

Outputs → `results/hierarchical_pyem/prolific_v1/`.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

# Pick up source edits without restarting the kernel.
for _k in list(sys.modules):
    if _k == "rlmodels_k99" or _k.startswith("rlmodels_k99."):
        del sys.modules[_k]

from rlmodels_k99.parameters import MODEL_PARAM_NAMES
from rlmodels_k99.registry import PRIMARY_MODEL_ORDER, get_model
from rlmodels_k99.recovery import compute_finalist_bic
from rlmodels_k99.behavior import (
    attach_rw_rpe_new,
    attach_model_subject_params,
    preprocess_combined_df,
    preprocess_learn_df,
    trim_combined_df_for_lme4,
)
from rlmodels_k99.data import learn_df_to_pyem
from rlmodels_k99.estimation import fit_hierarchical, production_recovery_fit_kwargs
from pyem.utils.math import alpha2norm, beta2norm

# --- knobs ---
NTRIALS = 80
N_STARTS = 3
BIC_NSAMPLES = 500
SHOW_PLOTS = True
PUB_DPI = 300
CONDITIONS = ("day_1",)

from membandit import paths
LOAD_DIR = Path(paths.get_prolific_dir())
OUT_DIR = Path(paths.results_root()) / "hierarchical_pyem" / "prolific_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": PUB_DPI,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

MODELS = list(PRIMARY_MODEL_ORDER)
print({"OUT_DIR": str(OUT_DIR), "models": MODELS, "n_models": len(MODELS)})


## Methods note (recovery not re-run)

Parameter and model recovery are **simulation-based** and depend only on task structure (80 trials, same five-block reversal schedule as EMU). That structure is identical here, so recovery eligibility and confusion-matrix methods are reported in `hierarchical_modeling.ipynb` (EMU) and are **not re-run** in this notebook.

This notebook fits the six primary models directly to Prolific learning data, compares them by integrated BIC (`BICint` / LME), and runs recognition-memory `p(hit)` regressions **separately** for `day_1` (immediate) and `day_2` (24 h delayed), with `zpm` = Gelman-standardized **image memorability** (`image_dprime`) within participant (not subject-level recognition d′).

## Load and normalize Prolific tables

Loads `learn_df_prolific_v2.csv` / `full_data_df_prolific_v2.csv` (from `prolific_preprocessing.ipynb`). Prolific CSVs use unsuffixed column names (`learn_trials`, `block`, `reward`, `dprime`, `image_dprime`) rather than EMU merge-suffixed names. We rename inline for `preprocess_*` / `attach_*` helpers and filter memory rows to `good_dm_performance` (learn-based flag).

**Important:** subject-level recognition `dprime` is kept as `subject_dprime`. Trial-level `image_dprime` is the memorability predictor used for `zpm`.


In [ ]:
# Corrected tables from prolific_preprocessing.ipynb (good_dm from learn rewards only).
learn_path = LOAD_DIR / "learn_df_prolific_v2.csv"
combined_path = LOAD_DIR / "full_data_df_prolific_v2.csv"

learn_df = None
combined_df = None
bundle = None

if learn_path.exists():
    learn_raw = pd.read_csv(learn_path)
    drop_cols = [c for c in learn_raw.columns if str(c).startswith("Unnamed")]
    if drop_cols:
        learn_raw = learn_raw.drop(columns=drop_cols)
    if "learn_trials" in learn_raw.columns and "trials" not in learn_raw.columns:
        learn_raw = learn_raw.rename(columns={"learn_trials": "trials"})
    learn_df = preprocess_learn_df(learn_raw)
    bundle = learn_df_to_pyem(learn_df, n_trials=NTRIALS)
    print(
        f"Loaded+preprocessed learn_df ({learn_path.name}): "
        f"{len(bundle.subject_ids)} subjects, {bundle.n_trials} trials "
        f"(good_dm filter applied by adapter)"
    )
else:
    print(f"No learn_df at {learn_path}")

if combined_path.exists():
    comb_raw = pd.read_csv(combined_path)
    drop_cols = [c for c in comb_raw.columns if str(c).startswith("Unnamed")]
    if drop_cols:
        comb_raw = comb_raw.drop(columns=drop_cols)
    # Keep all subjects in the CSV; analyses use good_dm_performance == True.
    if "good_dm_performance" in comb_raw.columns:
        n_before = comb_raw["participant"].nunique()
        comb_raw = comb_raw.loc[comb_raw["good_dm_performance"].astype(bool)].copy()
        print(
            f"combined_df good_dm filter: {comb_raw['participant'].nunique()} / {n_before} subjects retained"
        )
    rename = {}
    if "learn_trials" in comb_raw.columns and "trials_gamble" not in comb_raw.columns:
        rename["learn_trials"] = "trials_gamble"
    if "block" in comb_raw.columns and "block_gamble" not in comb_raw.columns:
        rename["block"] = "block_gamble"
    if "reward" in comb_raw.columns and "reward_gamble" not in comb_raw.columns:
        rename["reward"] = "reward_gamble"
    # Keep subject-level recognition d' separate from image memorability.
    if "dprime" in comb_raw.columns and "subject_dprime" not in comb_raw.columns:
        rename["dprime"] = "subject_dprime"
    if rename:
        comb_raw = comb_raw.rename(columns=rename)
        print("renamed combined columns:", rename)
    if "image_dprime" not in comb_raw.columns:
        raise ValueError("Prolific combined table missing image_dprime (needed for zpm / pm)")
    combined_df = preprocess_combined_df(comb_raw)
    # Do not let preprocess overwrite zpm from a missing DPRIME; build later from image_dprime.
    if "zpm" in combined_df.columns:
        combined_df = combined_df.drop(columns=["zpm"])
    cond_counts = (
        combined_df.groupby("condition")["participant"].nunique().to_dict()
        if "condition" in combined_df.columns
        else {}
    )
    print(
        f"Loaded+preprocessed combined_df ({combined_path.name}): "
        f"{len(combined_df)} rows, {combined_df['participant'].nunique()} subjects; "
        f"conditions={sorted(combined_df['condition'].dropna().unique().tolist()) if 'condition' in combined_df.columns else []}; "
        f"n_by_condition={cond_counts}"
    )
else:
    print(f"No combined_df at {combined_path}")

if learn_df is not None and combined_df is not None:
    inter = set(bundle.subject_ids) & set(combined_df["participant"].unique())
    print(f"learn ∩ memory subjects: {len(inter)} (learn={len(bundle.subject_ids)}, memory={combined_df['participant'].nunique()})")


In [ ]:
combined_df.keys()

## Fit primary models to Prolific learning data

Fit all six `PRIMARY_MODEL_ORDER` models with the same production EM kwargs used for EMU (no promotion gate — recovery referenced from the EMU notebook). **Slow cell:** 200 subjects × 6 models × integrated BIC Monte Carlo.


In [ ]:
real_fit_df = pd.DataFrame()
if bundle is None:
    print("skip real-data fit (no data)")
else:
    rows = []
    for name in MODELS:
        print("fitting:", name)
        fit_kw = production_recovery_fit_kwargs(name, seed=0)
        out = compute_finalist_bic(
            name,
            bundle.all_data,
            ntrials=bundle.n_trials,
            fit_kwargs=fit_kw,
            bic_nsamples=BIC_NSAMPLES,
            require_convergence=False,
        )
        rows.append(
            {
                "model": out["model"],
                "LME": out["LME"],
                "BICint": out["BICint"],
                "convergence": out["convergence"],
                "passed_gates": out["passed_gates"],
                "good_hess_frac": out["good_hess_frac"],
                "boundary_frac": out["boundary_frac"],
            }
        )
        params = pd.DataFrame(out["diagnostics"].subject_params, columns=MODEL_PARAM_NAMES[name])
        params.insert(0, "subject", bundle.subject_ids)
        params.to_csv(OUT_DIR / f"real_params_{name}.csv", index=False)
        del out

    real_fit_df = pd.DataFrame(rows).sort_values("LME", ascending=False)
    real_fit_df.to_csv(OUT_DIR / "real_data_model_comparison.csv", index=False)
    display(real_fit_df)

## Model comparison by integrated BIC

Publication ΔBIC bar plot (primary models only; winning model highlighted). Kass–Raftery guidelines at ΔBIC = 2 / 6 / 10.


In [ ]:
# Publication ΔBIC bar plot (primary models only; winning model highlighted).
cmp_path = OUT_DIR / "real_data_model_comparison.csv"
if "real_fit_df" in globals() and real_fit_df is not None and len(real_fit_df):
    bic_src = real_fit_df.copy()
elif cmp_path.exists():
    bic_src = pd.read_csv(cmp_path)
    print(f"loaded BICint table from {cmp_path}")
else:
    bic_src = pd.DataFrame()
    print("skip ΔBIC plot: no real_fit_df / comparison CSV")

if len(bic_src):
    primary = list(PRIMARY_MODEL_ORDER)
    bic_plot = (
        bic_src.loc[bic_src["model"].isin(primary)]
        .dropna(subset=["BICint"])
        .copy()
    )
    bic_plot = bic_plot.loc[np.isfinite(bic_plot["BICint"])].copy()
    if len(bic_plot) == 0:
        print("skip ΔBIC plot: no finite BICint after filtering")
    else:
        best = float(bic_plot["BICint"].min())
        bic_plot["delta_BIC"] = bic_plot["BICint"] - best
        bic_plot = bic_plot.sort_values("delta_BIC", ascending=True).reset_index(drop=True)
        assert set(bic_plot["model"]).issubset(set(primary))
        assert np.isclose(bic_plot["delta_BIC"].min(), 0.0)
        assert np.isfinite(bic_plot["delta_BIC"]).all()
        display(bic_plot[["model", "LME", "BICint", "delta_BIC"]])

        fig, ax = plt.subplots(figsize=(7.2, 4.0))
        colors = ["#2f6fed" if i == 0 else "#9aa3af" for i in range(len(bic_plot))]
        bars = ax.bar(bic_plot["model"], bic_plot["delta_BIC"], color=colors, edgecolor="white", width=0.72)
        for bar, val in zip(bars, bic_plot["delta_BIC"]):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val + max(float(bic_plot["delta_BIC"].max()) * 0.02, 0.4),
                f"{val:.1f}",
                ha="center",
                va="bottom",
                fontsize=10,
            )
        ymax = max(float(bic_plot["delta_BIC"].max()) * 1.18, 12.0)
        for y, label in [(2, "weak"), (6, "positive"), (10, "strong")]:
            if y < ymax:
                ax.axhline(y, color="#c7ccd1", ls="--", lw=1.0, zorder=0)
                ax.text(len(bic_plot) - 0.45, y + 0.15, label, ha="right", va="bottom", fontsize=8, color="#6b7280")
        ax.set_ylabel(r"$\Delta$BIC (integrated; best = 0)")
        ax.set_xlabel("model")
        ax.set_title("Prolific: hierarchical model comparison (integrated $\Delta$BIC)")
        ax.set_ylim(0, ymax)
        fig.tight_layout()

        out_csv = OUT_DIR / "model_comparison_delta_bic.csv"
        out_pdf = OUT_DIR / "model_comparison_delta_bic.pdf"
        bic_plot.to_csv(out_csv, index=False)
        fig.savefig(out_pdf, dpi=PUB_DPI, bbox_inches="tight")
        if SHOW_PLOTS:
            plt.show()
        else:
            plt.close(fig)
        print("wrote", out_csv)
        print("wrote", out_pdf)
        print(f"best model: {bic_plot.iloc[0]['model']} (BICint={bic_plot.iloc[0]['BICint']:.1f})")


## Attach hierarchical RW α, β, and trial RPE; re-save dataframes

Use subject parameters from `real_params_rw.csv` (the production fit above) to simulate trial-wise RPE on each subject's choices/rewards. Write `alpha` / `beta` / `rpe` onto `learn_df` and `combined_df`, then overwrite the v2 CSVs on `LOAD_DIR` so later recognition / comparison cells can use them.


In [ ]:
# Attach hierarchical RW alpha/beta/rpe from the production real_params_rw fit, then re-save v2 CSVs.
rw_params_path = OUT_DIR / "real_params_rw.csv"

if bundle is None or learn_df is None:
    print("skip RW attach+save: need learn_df + pyEM bundle")
elif not rw_params_path.exists():
    print(f"skip RW attach+save: missing {rw_params_path} (run the primary-model fit cell first)")
else:
    rw_params = (
        pd.read_csv(rw_params_path)
        .rename(columns={"subject": "participant"})[["participant", "alpha", "beta"]]
        .drop_duplicates("participant", keep="last")
        .copy()
    )
    # Align to fitted subject order
    rw_params = rw_params.set_index("participant").reindex(bundle.subject_ids).reset_index()
    if rw_params[["alpha", "beta"]].isna().any().any():
        missing = rw_params.loc[rw_params[["alpha", "beta"]].isna().any(axis=1), "participant"].tolist()
        raise ValueError(f"real_params_rw missing alpha/beta for {len(missing)} subjects: {missing[:5]}")

    # Trial-wise RPE from fitted subject params (no second hierarchical EM).
    trial_frames = []
    param_rows = []
    for i, pid in enumerate(bundle.subject_ids):
        alpha_hat = float(rw_params.loc[i, "alpha"])
        beta_hat = float(rw_params.loc[i, "beta"])
        param_rows.append(
            {
                "participant": pid,
                "model": "rw",
                "alpha_fit": alpha_hat,
                "beta_fit": beta_hat,
            }
        )
        raw = np.array([beta2norm(beta_hat), alpha2norm(alpha_hat)], dtype=float)
        pe = np.asarray(
            get_model("rw").fit(raw, bundle.choices[i], bundle.rewards[i], output="all")["PE"],
            dtype=float,
        )
        trial_frames.append(
            pd.DataFrame(
                {
                    "participant": pid,
                    "trials": np.arange(1, len(pe) + 1),
                    "model": "rw",
                    "rpe_fit": pe,
                    "alpha_fit": alpha_hat,
                    "beta_fit": beta_hat,
                }
            )
        )

    param_cmp = pd.DataFrame(param_rows)
    trial_cmp = pd.concat(trial_frames, ignore_index=True)
    param_cmp.to_csv(OUT_DIR / "learn_df_param_comparison_rw.csv", index=False)
    trial_cmp.to_csv(OUT_DIR / "learn_df_rpe_comparison_rw.csv", index=False)

    # --- learn_df: alpha / beta / rpe ---
    tcol = "trials" if "trials" in learn_df.columns else "learn_trials"
    learn_df = learn_df.drop(columns=["alpha", "beta", "rpe"], errors="ignore")
    learn_df = learn_df.merge(
        rw_params[["participant", "alpha", "beta"]],
        on="participant",
        how="left",
    )
    learn_df = learn_df.merge(
        trial_cmp[["participant", "trials", "rpe_fit"]].rename(
            columns={"trials": tcol, "rpe_fit": "rpe"}
        ),
        on=["participant", tcol],
        how="left",
    )
    n_miss_learn = int(learn_df.loc[learn_df["participant"].isin(bundle.subject_ids), ["alpha", "beta", "rpe"]].isna().any(axis=1).sum())
    if n_miss_learn:
        raise ValueError(f"learn_df still has {n_miss_learn} rows missing alpha/beta/rpe after merge")

    # --- combined_df: same columns via trials_gamble ---
    if combined_df is not None and len(combined_df):
        combined_df = combined_df.drop(columns=["alpha", "beta", "rpe"], errors="ignore")
        combined_df = combined_df.merge(
            rw_params[["participant", "alpha", "beta"]],
            on="participant",
            how="left",
        )
        # attach_rw_rpe_new also creates rpe_new / zrpe_new for recognition cells
        combined_df = attach_rw_rpe_new(combined_df, trial_cmp, model="rw")
        combined_df = attach_model_subject_params(
            combined_df, param_cmp, model="rw", recompute_zpm=False, standardize_learn_df_params=True
        )
        # Canonical rpe column (encoding-trial RPE) for drop-in use
        if "rpe_new" in combined_df.columns:
            combined_df["rpe"] = combined_df["rpe_new"]
        n_studied = combined_df["trials_gamble"].notna() if "trials_gamble" in combined_df.columns else pd.Series(False, index=combined_df.index)
        n_miss_rpe = int(combined_df.loc[n_studied, "rpe"].isna().sum()) if "rpe" in combined_df.columns else 0
        print(
            f"combined_df: subjects={combined_df['participant'].nunique()} | "
            f"studied rows missing rpe={n_miss_rpe}"
        )
    else:
        print("combined_df not loaded; saved learn_df only")

    # Re-save v2 tables (trial index column name as on disk: learn_trials)
    learn_save = learn_df.copy()
    if "trials" in learn_save.columns and "learn_trials" not in learn_save.columns:
        learn_save = learn_save.rename(columns={"trials": "learn_trials"})
    learn_out = LOAD_DIR / "learn_df_prolific_v2.csv"
    learn_save.to_csv(learn_out, index=False)
    print(f"wrote {learn_out} | subjects={learn_save['participant'].nunique()} | rows={len(learn_save)}")

    if combined_df is not None and len(combined_df):
        comb_save = combined_df.copy()
        # Disk schema uses unsuffixed names (matching prolific_preprocessing)
        rev = {}
        if "trials_gamble" in comb_save.columns and "learn_trials" not in comb_save.columns:
            rev["trials_gamble"] = "learn_trials"
        if "block_gamble" in comb_save.columns and "block" not in comb_save.columns:
            rev["block_gamble"] = "block"
        if "reward_gamble" in comb_save.columns and "reward" not in comb_save.columns:
            rev["reward_gamble"] = "reward"
        if "subject_dprime" in comb_save.columns and "dprime" not in comb_save.columns:
            rev["subject_dprime"] = "dprime"
        if rev:
            comb_save = comb_save.rename(columns=rev)
        # Keep LOAD_DIR schema lean: drop modeling-only *_new / z* columns from the v2 table
        drop_model = [
            c for c in comb_save.columns
            if c.endswith("_new") or c.startswith("zalpha") or c.startswith("zbeta")
            or c in {"zrpe_new", "zrpe"}
        ]
        if drop_model:
            comb_save = comb_save.drop(columns=drop_model, errors="ignore")
        full_out = LOAD_DIR / "full_data_df_prolific_v2.csv"
        comb_save.to_csv(full_out, index=False)
        combined_df.to_csv(OUT_DIR / "combined_df_with_rpe_new.csv", index=False)
        print(f"wrote {full_out} | subjects={comb_save['participant'].nunique()} | rows={len(comb_save)}")
        print("wrote", OUT_DIR / "combined_df_with_rpe_new.csv")

    print("wrote", OUT_DIR / "learn_df_param_comparison_rw.csv")
    print("wrote", OUT_DIR / "learn_df_rpe_comparison_rw.csv")
    display(rw_params.head())


## Compute Trial-Wise RPE and Subject Parameters Under Hierarchical RW

Extract subject-level learning rate ($\alpha$), inverse temperature ($\beta$), and trial-level RPE ($\delta_t$) from the hierarchical `rw` model.


In [ ]:
COMPARE_MODELS = ["rw"]

if bundle is None:
    print("skip learn_df comparison (no data)")
else:
    ref_subj = (
        learn_df.loc[learn_df["participant"].isin(bundle.subject_ids), ["participant", "alpha", "beta"]]
        .drop_duplicates("participant")
        .set_index("participant")
        .reindex(bundle.subject_ids)
    )
    ref_trials = (
        learn_df.loc[
            learn_df["participant"].isin(bundle.subject_ids),
            ["participant", "trials", "rpe", "alpha", "beta"],
        ]
        .sort_values(["participant", "trials"], kind="mergesort")
        .reset_index(drop=True)
    )

    param_rows, trial_frames, fit_store = [], [], {}
    for name in COMPARE_MODELS:
        print(f"fitting hierarchical {name} for learn_df comparison...")
        _em, diag = fit_hierarchical(
            name,
            bundle.all_data,
            fit_kwargs=production_recovery_fit_kwargs(name, seed=0),
            n_starts=N_STARTS,
            require_convergence=False,
        )
        fit_store[name] = diag
        est_df = pd.DataFrame(np.asarray(diag.subject_params, dtype=float), columns=MODEL_PARAM_NAMES[name])
        est_df.insert(0, "participant", bundle.subject_ids)

        for i, pid in enumerate(bundle.subject_ids):
            beta_hat = float(est_df.loc[i, "beta"])
            alpha_hat = float(est_df.loc[i, "alpha"])
            param_rows.append(
                {
                    "participant": pid,
                    "model": name,
                    "alpha_learn_df": float(ref_subj.loc[pid, "alpha"]),
                    "beta_learn_df": float(ref_subj.loc[pid, "beta"]),
                    "alpha_fit": alpha_hat,
                    "beta_fit": beta_hat,
                    "convergence": bool(diag.convergence),
                    "lme": float(diag.lme),
                }
            )
            raw = np.array([beta2norm(beta_hat), alpha2norm(alpha_hat)], dtype=float)
            pe = np.asarray(
                get_model(name).fit(raw, bundle.choices[i], bundle.rewards[i], output="all")["PE"],
                dtype=float,
            )
            trial_frames.append(
                pd.DataFrame(
                    {
                        "participant": pid,
                        "trials": np.arange(1, len(pe) + 1),
                        "model": name,
                        "rpe_fit": pe,
                        "alpha_fit": alpha_hat,
                        "beta_fit": beta_hat,
                    }
                )
            )

    param_cmp = pd.DataFrame(param_rows)
    trial_cmp = pd.concat(trial_frames, ignore_index=True).merge(
        ref_trials, on=["participant", "trials"], how="inner"
    )
    param_cmp.to_csv(OUT_DIR / "learn_df_param_comparison_rw.csv", index=False)
    trial_cmp.to_csv(OUT_DIR / "learn_df_rpe_comparison_rw.csv", index=False)

    real_rw_path = OUT_DIR / "real_params_rw.csv"
    if real_rw_path.exists():
        real_rw = pd.read_csv(real_rw_path)
        display(real_rw.head())
    display(param_cmp.head())
    print("wrote", OUT_DIR / "learn_df_param_comparison_rw.csv")
    print("wrote", OUT_DIR / "learn_df_rpe_comparison_rw.csv")

## Within-Participant RPE Tercile Summary

Within-participant tercile categories (`negative` / `neutral` / `positive`) for legacy `learn_df` RPE vs hierarchical RW `rpe_fit`. Exact agreement and Cohen's κ quantify relative-rank alignment, not raw magnitude equality.


In [ ]:
# Within-participant RPE tercile alignment (legacy rpe vs hierarchical RW rpe_fit).
RPE_CATS = ["negative", "neutral", "positive"]


def _cohen_kappa(y_true, y_pred, labels):
    """Unweighted Cohen's kappa without sklearn."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    labels = list(labels)
    k = len(labels)
    idx = {lab: i for i, lab in enumerate(labels)}
    cm = np.zeros((k, k), dtype=float)
    for a, b in zip(y_true, y_pred):
        if a in idx and b in idx:
            cm[idx[a], idx[b]] += 1.0
    n = cm.sum()
    if n <= 0:
        return float("nan")
    po = np.trace(cm) / n
    pe = float(np.dot(cm.sum(axis=1), cm.sum(axis=0)) / (n * n))
    if np.isclose(pe, 1.0):
        return float("nan")
    return float((po - pe) / (1.0 - pe))


def _within_subj_terciles(series: pd.Series) -> pd.Series:
    out = pd.Series(index=series.index, dtype="object")
    vals = pd.to_numeric(series, errors="coerce")
    valid = vals.dropna()
    if len(valid) < 3:
        return out
    try:
        cats = pd.qcut(valid, q=3, labels=RPE_CATS, duplicates="drop")
        out.loc[valid.index] = cats.astype(str)
    except Exception:
        return out
    return out


align_src = None
if "trial_cmp" in globals() and trial_cmp is not None and len(trial_cmp):
    align_df = trial_cmp.loc[trial_cmp["model"] == "rw"].copy()
    align_src = "trial_cmp"
else:
    _path = OUT_DIR / "learn_df_rpe_comparison_rw.csv"
    if _path.exists():
        align_df = pd.read_csv(_path)
        if "model" in align_df.columns:
            align_df = align_df.loc[align_df["model"] == "rw"].copy()
        align_src = str(_path)
    else:
        align_df = pd.DataFrame()
        print("skip tercile alignment: no trial_cmp / learn_df_rpe_comparison_rw.csv")

if len(align_df):
    need = {"participant", "trials", "rpe", "rpe_fit"}
    missing = need - set(align_df.columns)
    if missing:
        raise ValueError(f"tercile alignment missing columns: {sorted(missing)}")

    align_df = align_df.sort_values(["participant", "trials"], kind="mergesort").reset_index(drop=True)
    align_df["rpe_cat_learn"] = (
        align_df.groupby("participant", group_keys=False)["rpe"].apply(_within_subj_terciles)
    )
    align_df["rpe_cat_rw"] = (
        align_df.groupby("participant", group_keys=False)["rpe_fit"].apply(_within_subj_terciles)
    )
    x = align_df.dropna(subset=["rpe_cat_learn", "rpe_cat_rw"]).copy()
    x["agree"] = x["rpe_cat_learn"].astype(str) == x["rpe_cat_rw"].astype(str)

    n_trials = int(len(x))
    n_subj = int(x["participant"].nunique())
    agreement = float(x["agree"].mean()) if n_trials else float("nan")
    kappa = _cohen_kappa(x["rpe_cat_learn"], x["rpe_cat_rw"], RPE_CATS) if n_trials else float("nan")

    cm = (
        pd.crosstab(x["rpe_cat_learn"], x["rpe_cat_rw"])
        .reindex(index=RPE_CATS, columns=RPE_CATS, fill_value=0)
    )
    cm_norm = cm.div(cm.sum(axis=1).replace(0, np.nan), axis=0)

    by_subj = (
        x.groupby("participant", sort=False)
        .agg(
            n_trials=("agree", "size"),
            agreement=("agree", "mean"),
            n_learn_neg=("rpe_cat_learn", lambda s: int((s == "negative").sum())),
            n_rw_neg=("rpe_cat_rw", lambda s: int((s == "negative").sum())),
        )
        .reset_index()
    )
    by_subj["kappa"] = [
        _cohen_kappa(
            x.loc[x["participant"] == pid, "rpe_cat_learn"],
            x.loc[x["participant"] == pid, "rpe_cat_rw"],
            RPE_CATS,
        )
        for pid in by_subj["participant"]
    ]

    summary = pd.DataFrame(
        [
            {
                "n_trials": n_trials,
                "n_subjects": n_subj,
                "exact_agreement": agreement,
                "cohens_kappa": kappa,
                "subj_agreement_mean": float(by_subj["agreement"].mean()),
                "subj_agreement_median": float(by_subj["agreement"].median()),
                "subj_agreement_min": float(by_subj["agreement"].min()),
                "subj_agreement_max": float(by_subj["agreement"].max()),
                "source": align_src,
            }
        ]
    )
    display(summary)
    display(cm)
    display(cm_norm.round(3))
    print(
        f"exact agreement={agreement:.3%} | Cohen κ={kappa:.3f} | "
        f"n={n_trials} trials / {n_subj} subjects"
    )

    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    im = ax.imshow(cm_norm.to_numpy(dtype=float), vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(RPE_CATS)))
    ax.set_yticks(range(len(RPE_CATS)))
    ax.set_xticklabels(RPE_CATS)
    ax.set_yticklabels(RPE_CATS)
    ax.set_xlabel("hierarchical RW tercile")
    ax.set_ylabel("legacy learn_df tercile")
    ax.set_title("Prolific: within-participant RPE tercile alignment")
    for i in range(len(RPE_CATS)):
        for j in range(len(RPE_CATS)):
            count = int(cm.to_numpy()[i, j])
            frac = cm_norm.to_numpy(dtype=float)[i, j]
            if np.isfinite(frac):
                ax.text(
                    j,
                    i,
                    f"{count}\n({frac:.0%})",
                    ha="center",
                    va="center",
                    fontsize=10,
                    color="white" if frac > 0.55 else "black",
                )
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="row-normalized fraction")
    fig.tight_layout()

    out_summary = OUT_DIR / "rw_rpe_tercile_alignment_summary.csv"
    out_subj = OUT_DIR / "rw_rpe_tercile_alignment_by_subject.csv"
    out_cm = OUT_DIR / "rw_rpe_tercile_alignment_confusion.csv"
    out_pdf = OUT_DIR / "rw_rpe_tercile_alignment.pdf"
    summary.to_csv(out_summary, index=False)
    by_subj.to_csv(out_subj, index=False)
    cm.to_csv(out_cm)
    fig.savefig(out_pdf, dpi=PUB_DPI, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
    print("wrote", out_summary)
    print("wrote", out_subj)
    print("wrote", out_cm)
    print("wrote", out_pdf)


## Publication figure: RW α–β joint distribution

Joint density of hierarchical Rescorla–Wagner subject parameters (learning rate vs inverse temperature) from the Prolific fit.


In [ ]:
# Prefer empirical hierarchical RW fits; fall back to learn_df comparison output.
rw_params = None
real_rw_path = OUT_DIR / "real_params_rw.csv"
if real_rw_path.exists():
    rw_params = pd.read_csv(real_rw_path)[["alpha", "beta"]].dropna()
    src_label = "hierarchical rw (real_params_rw)"
elif "param_cmp" in globals() and param_cmp is not None and len(param_cmp):
    rw_params = (
        param_cmp.loc[param_cmp["model"] == "rw", ["alpha_fit", "beta_fit"]]
        .rename(columns={"alpha_fit": "alpha", "beta_fit": "beta"})
        .dropna()
    )
    src_label = "hierarchical rw (param_cmp)"
else:
    print("skip α–β joint plot: no RW parameters available")

if rw_params is not None and len(rw_params):
    g = sns.jointplot(
        data=rw_params,
        x="alpha",
        y="beta",
        kind="kde",
        fill=True,
        cmap="Spectral_r",
        thresh=0.05,
        levels=12,
        height=5.2,
        space=0,
        joint_kws=dict(bw_adjust=0.5),
        marginal_kws=dict(fill=True, color="#b8a9c9", alpha=0.7, bw_adjust=0.5),
    )

    g.ax_joint.set_xlabel(r"RW $\alpha$")
    g.ax_joint.set_ylabel(r"RW $\beta$")
    g.ax_joint.set_xlim(-0.5, 1.5)
    g.ax_joint.set_ylim(-1, 12)
    g.figure.suptitle(r"Prolific: joint distribution of RW $\alpha$ and $\beta$", y=1.02)
    if g.ax_joint.collections:
        cbar = g.figure.colorbar(
            g.ax_joint.collections[-1], ax=g.ax_joint, orientation="horizontal", pad=0.12, fraction=0.05
        )
        cbar.set_label("density")
    # Overlay subject points so KDE tails beyond [0, 1] aren't mistaken for data.
    # g.ax_joint.scatter(
    #     rw_params["alpha"],
    #     rw_params["beta"],
    #     s=28,
    #     c="black",
    #     alpha=0.9,
    #     zorder=5,
    #     edgecolors="none",
    # )
    out_path = OUT_DIR / "rw_alpha_beta_joint.pdf"
    g.figure.savefig(out_path, dpi=PUB_DPI, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(g.figure)
    print(f"wrote {out_path} ({src_label}; n={len(rw_params)})")


## Recognition memory: P(hit) ~ hierarchical RW predictors (per condition)

Attach hierarchical RW trial RPE (`rpe_new` / `zrpe_new`) and subject α/β (`alpha_new` / `beta_new`). Fit **separately** for `day_1` and `day_2`. Within each condition, set `zpm` from trial-level **`image_dprime`** via within-participant Gelman standardization `(x - mean) / (2 * SD)`. Export lean CSVs for R `glmer`.

Run the R block in the next markdown cell after this Python export completes.


In [ ]:
if combined_df is None:
    print("skip recognition export: combined_df not loaded")
elif "trial_cmp" not in globals() or trial_cmp is None or len(trial_cmp) == 0:
    print("skip recognition export: run the learn_df comparison cell first (needs trial_cmp)")
elif "param_cmp" not in globals() or param_cmp is None or len(param_cmp) == 0:
    print("skip recognition export: run the learn_df comparison cell first (needs param_cmp)")
else:
    combined_df = attach_rw_rpe_new(combined_df, trial_cmp, model="rw")
    combined_df = attach_model_subject_params(
        combined_df, param_cmp, model="rw", recompute_zpm=False
    )
    combined_df.to_csv(OUT_DIR / "combined_df_with_rpe_new.csv", index=False)

    for cond in CONDITIONS:
        sub = combined_df.loc[combined_df["condition"] == cond].copy()
        # Exclude participants at or below chance on recognition memory (d' < 0; n = 193 retained)
        dcol = "subject_dprime" if "subject_dprime" in sub.columns else ("dprime" if "dprime" in sub.columns else None)
        if dcol is not None:
            sub = sub.loc[pd.to_numeric(sub[dcol], errors="coerce") >= 0].copy()
            if cond == "day_1":
                assert sub["participant"].nunique() == 193, f"Expected 193 online Day-1 memory subjects, got {sub['participant'].nunique()}"
        if len(sub) == 0:
            print(f"skip {cond}: no rows")
            continue

        # Within-participant Gelman zpm from image memorability (image_dprime).
        if "image_dprime" not in sub.columns:
            print(f"skip {cond}: no image_dprime column")
            continue
        sub["zpm"] = np.nan
        for pid, g in sub.groupby("participant", sort=False):
            vals = pd.to_numeric(g["image_dprime"], errors="coerce")
            mu = float(np.nanmean(vals.to_numpy(dtype=float)))
            sd = float(np.nanstd(vals.to_numpy(dtype=float)))
            if np.isfinite(sd) and sd > 0:
                sub.loc[g.index, "zpm"] = (vals - mu) / (2.0 * sd)

        model_df = trim_combined_df_for_lme4(sub, studied_only=True, require_rpe_new=True)
        model_df["hit"] = model_df["hit"].astype(float)
        # Ensure image_dprime survives if present (for QC); zpm is required.
        if "image_dprime" in sub.columns and "image_dprime" not in model_df.columns:
            model_df = model_df.merge(
                sub[["participant", "trials_gamble", "image_dprime"]].drop_duplicates(),
                on=["participant", "trials_gamble"],
                how="left",
            )
        zpm_nuniq = (
            model_df.groupby("participant")["zpm"].nunique().median()
            if "zpm" in model_df.columns
            else 0
        )
        print(
            f"{cond}: n={len(model_df)} rows; "
            f"subjects={model_df['participant'].nunique()}; "
            f"median within-subj unique zpm={zpm_nuniq}"
        )
        assert zpm_nuniq > 1, f"{cond}: zpm looks subject-constant; expected trial-varying image_dprime zpm"

        needed = {"zrpe_new", "zpm", "zalpha_new", "zbeta_new", "hit", "participant"}
        missing = needed - set(model_df.columns)
        if missing:
            raise ValueError(f"{cond}: trimmed combined_df missing columns: {sorted(missing)}")

        lme4_path = OUT_DIR / f"combined_df_lme4_hit_rpe_new_{cond}.csv"
        model_df.to_csv(lme4_path, index=False)
        print("wrote", lme4_path)

    print("Next: run the R markdown cell below, then the coefficient-plot cell.")
    print("Expected R outputs:")
    for name in [
        "glmer_hit_coefficients_day_1.csv",
        "glmer_hit_coefficients_day_1_core.csv",
    ]:
        print(" ", OUT_DIR / name)


## R: Day-1 binomial `glmer` (Prolific)

Copy/paste into an R session after the Python export above. Fits demographic-adjusted Day-1/Day-2 models for within-cohort figures, plus a Day-1 **core** model (no age/sex) for cross-cohort comparison. Requires package `lme4` (no `broom`).

`zpm` in the CSVs is already Gelman-standardized `image_dprime`.

```r
library(lme4)

# Run from repository root via: Rscript R/glmer_recognition_online.R
base <- "results/hierarchical_pyem/prolific_v1"
data_dir <- file.path(Sys.getenv("MEMORYBANDIT_DATA_DIR"), "work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest")

d1 <- read.csv(file.path(base, "combined_df_lme4_hit_rpe_new_day_1.csv"))
d2 <- read.csv(file.path(base, "combined_df_lme4_hit_rpe_new_day_2.csv"))
dem <- read.csv(file.path(data_dir, "demographics.csv"))

# Some demographic rows are shifted:
# normal: Age=33, Sex=Male
# shifted: Age=Yes/No, Sex=33, Ethnicity.simplified=Male
age_from_age <- suppressWarnings(as.numeric(as.character(dem$Age)))
age_from_sex <- suppressWarnings(as.numeric(as.character(dem$Sex)))

dem$age <- ifelse(!is.na(age_from_age), age_from_age, age_from_sex)

dem$sex <- ifelse(
  dem$Sex %in% c("Female", "Male"),
  as.character(dem$Sex),
  ifelse(
    dem$Ethnicity.simplified %in% c("Female", "Male"),
    as.character(dem$Ethnicity.simplified),
    NA_character_
  )
)

# Binary coding before standardization: Female=0, Male=1
dem$sex_binary <- ifelse(
  dem$sex == "Male", 1,
  ifelse(dem$sex == "Female", 0, NA_real_)
)

demographics <- data.frame(
  participant = as.character(dem$Participant.id),
  age = dem$age,
  sex_binary = dem$sex_binary
)
demographics <- demographics[!duplicated(demographics$participant), ]

analysis_ids <- unique(c(as.character(d1$participant), as.character(d2$participant)))
demographics <- demographics[demographics$participant %in% analysis_ids, ]

gelman_standardize <- function(x) {
  (x - mean(x, na.rm = TRUE)) / (2 * sd(x, na.rm = TRUE))
}

demographics$zage <- gelman_standardize(demographics$age)
demographics$zsex <- gelman_standardize(demographics$sex_binary)

d1 <- merge(d1, demographics, by = "participant", all.x = TRUE)
d2 <- merge(d2, demographics, by = "participant", all.x = TRUE)

# QC: zpm should be trial-varying image memorability (~40 unique / subject)
stopifnot(median(tapply(d1$zpm, d1$participant, function(x) length(unique(x)))) > 1)

vars_core <- c("hit", "zrpe_new", "zpm", "zalpha_new", "zbeta_new", "participant")
vars_demo <- c(vars_core, "zage", "zsex")

# --- Day 1 demographic-adjusted (within-cohort figure) ---
d1_model <- d1[complete.cases(d1[, vars_demo]), ]
m_day1 <- glmer(
  hit ~ zrpe_new + zpm + zalpha_new + zbeta_new + zage + zsex + (1 | participant),
  data = d1_model,
  family = binomial(link = "logit"),
  control = glmerControl(optimizer = "bobyqa")
)
summary(m_day1)

# --- Day 2 demographic-adjusted ---
d2_model <- d2[complete.cases(d2[, vars_demo]), ]
m_day2 <- glmer(
  hit ~ zrpe_new + zpm + zalpha_new + zbeta_new + zage + zsex + (1 | participant),
  data = d2_model,
  family = binomial(link = "logit"),
  control = glmerControl(optimizer = "bobyqa")
)
summary(m_day2)

# --- Day 1 core (cross-cohort; no age/sex) ---
d1_core <- d1[complete.cases(d1[, vars_core]), ]
m_day1_core <- glmer(
  hit ~ zrpe_new + zpm + zalpha_new + zbeta_new + (1 | participant),
  data = d1_core,
  family = binomial(link = "logit"),
  control = glmerControl(optimizer = "bobyqa")
)
summary(m_day1_core)

export_fixed_effects <- function(model, path) {
  s <- coef(summary(model))
  out <- data.frame(
    term = rownames(s),
    estimate = s[, "Estimate"],
    std.error = s[, "Std. Error"],
    statistic = s[, "z value"],
    p.value = s[, "Pr(>|z|)"],
    row.names = NULL,
    check.names = FALSE
  )
  write.csv(out, path, row.names = FALSE)
}

export_fixed_effects(m_day1, file.path(base, "glmer_hit_coefficients_day_1.csv"))
export_fixed_effects(m_day2, file.path(base, "glmer_hit_coefficients_day_2.csv"))
export_fixed_effects(m_day1_core, file.path(base, "glmer_hit_coefficients_day_1_core.csv"))
```


## Publication figure: `glmer` coefficients for P(hit) (per condition)

Requires R `glmer` coefficient CSVs produced by the R markdown cell above. One panel/file per condition (`day_1`). Demographic-adjusted models include Gelman-standardized age (`zage`) and binary sex (`zsex`) when present in the R exports.


In [ ]:
from scipy.stats import norm

SIG_COLOR, NS_COLOR, ALPHA_SIG = "#2f6fed", "#6e6e6e", 0.05
LABELS = {
    "zrpe_new": r"$rpe$",
    "zpm": r"$pm$",
    "zalpha_new": r"RW$\alpha$",
    "zbeta_new": r"RW$\beta$",
    "zage": "Age",
    "zsex": "Sex",
    "(Intercept)": "Intercept",
    "Intercept": "Intercept",
}
ORDER = ["zalpha_new", "zpm", "zrpe_new", "zbeta_new", "zage", "zsex"]


def _load_coef_df(cond: str):
    glmer_csv = OUT_DIR / f"glmer_hit_coefficients_{cond}.csv"
    if not glmer_csv.exists():
        raise FileNotFoundError(
            f"Missing {glmer_csv}. Run the R markdown cell above to fit glmer and write "
            f"coefficient CSVs for day_1/day_2, then re-run this cell."
        )
    raw = pd.read_csv(glmer_csv)
    raw = raw.rename(columns={c: c.lower().replace(".", "_") for c in raw.columns})
    if not {"term", "estimate"}.issubset(raw.columns):
        raise ValueError(f"{glmer_csv.name}: need term/estimate columns; got {list(raw.columns)}")
    se_col = "std_error" if "std_error" in raw.columns else ("std_err" if "std_err" in raw.columns else None)
    p_col = "p_value" if "p_value" in raw.columns else None
    if se_col is None:
        raise ValueError(f"{glmer_csv.name}: missing std.error / std_err")
    coef_df = pd.DataFrame(
        {
            "name": raw["term"].astype(str),
            "coef": pd.to_numeric(raw["estimate"], errors="coerce"),
            "se": pd.to_numeric(raw[se_col], errors="coerce"),
            "p": pd.to_numeric(raw[p_col], errors="coerce") if p_col else np.nan,
        }
    )
    return coef_df, f"R glmer ({glmer_csv.name})"


for cond in CONDITIONS:
    coef_df, src = _load_coef_df(cond)
    rows = []
    for name in ORDER:
        hit = coef_df.loc[coef_df["name"] == name]
        if len(hit) == 0:
            continue
        r = hit.iloc[0]
        coef = float(r["coef"])
        se = float(r["se"]) if np.isfinite(r.get("se", np.nan)) else np.nan
        p = float(r["p"]) if "p" in r and np.isfinite(r.get("p", np.nan)) else np.nan
        sig = bool(np.isfinite(p) and p < ALPHA_SIG)
        rows.append(
            {
                "name": name,
                "label": LABELS.get(name, name),
                "coef": coef,
                "se": se,
                "p": p,
                "significant": sig,
                "color": SIG_COLOR if sig else NS_COLOR,
            }
        )
    if not rows:
        raise ValueError(f"{cond}: no matching fixed effects in glmer coefficient CSV")

    x_lo = min(r["coef"] - 3.0 * r["se"] for r in rows if np.isfinite(r["se"]))
    x_hi = max(r["coef"] + 3.0 * r["se"] for r in rows if np.isfinite(r["se"]))
    x_lo, x_hi = min(x_lo, 0.0), max(x_hi, 0.0)
    pad = 0.08 * (x_hi - x_lo if x_hi > x_lo else 1.0)
    x_lo, x_hi = x_lo - pad, x_hi + pad

    fig_height = max(3.6, 0.7 * len(rows) + 1.2)
    fig, ax = plt.subplots(figsize=(6.5, fig_height))
    y_positions = list(range(len(rows)))[::-1]
    x_grid = np.linspace(x_lo, x_hi, 500)
    for y, row in zip(y_positions, rows):
        if np.isfinite(row["se"]) and row["se"] > 0:
            dens = norm.pdf(x_grid, loc=row["coef"], scale=row["se"])
            dens = dens / dens.max() * 0.35
            ax.fill_between(x_grid, y, y + dens, color=row["color"], alpha=0.35, linewidth=0)
            lo, hi = row["coef"] - 1.96 * row["se"], row["coef"] + 1.96 * row["se"]
            ax.plot([lo, hi], [y, y], color=row["color"], lw=2.0, solid_capstyle="round")
        ax.plot(row["coef"], y, "o", color=row["color"], ms=7, zorder=3)
        ax.text(
            x_lo - 0.02 * (x_hi - x_lo),
            y,
            row["label"],
            ha="right",
            va="center",
            fontsize=12,
            color=row["color"] if row["significant"] else "black",
        )
    ax.axvline(0.0, color="black", ls="--", lw=1.0)
    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(-0.4, len(rows) - 0.2)
    ax.set_yticks([])
    ax.set_xlabel("model weight")
    ax.set_title(rf"$p(\mathrm{{hit}})\ \sim$ glmer fixed effects ({cond})", loc="left")
    ax.spines["left"].set_visible(False)
    fig.tight_layout()
    out_path = OUT_DIR / f"glmer_hit_coefficients_{cond}.pdf"
    fig.savefig(out_path, dpi=PUB_DPI, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
    print("wrote", out_path, f"({src})")
    display(pd.DataFrame(rows)[["label", "name", "coef", "se", "p", "significant"]])


## Notes

- Outputs write to `results/hierarchical_pyem/prolific_v1/`.
- Parameter / model recovery is **not** re-run here; see `hierarchical_modeling.ipynb` (EMU) — task structure is identical (80 trials, same reversals).
- Final manuscript set: `wsls`, `rw`, `rwf`, `rw_asym`, `bayes_fixed_reward`, `vkf_omega`.
- Recognition regressions are **per condition** (`day_1`) with within-participant Gelman `zpm` from `image_dprime` (not subject d′).
- Manuscript recognition models are R `lme4::glmer` only (see R markdown cell); coefficient plots require the R-written CSVs.
- Publication figures save as PDF at `PUB_DPI=300`.
- Key exports: `real_data_model_comparison.csv`, `model_comparison_delta_bic.pdf`, `rw_alpha_beta_joint.pdf`, `combined_df_lme4_hit_rpe_new_{day_1,day_2}.csv`, `glmer_hit_coefficients_{day_1,day_2}.pdf`.
- Cross-cohort Day-1 core `zrpe_new` comparison: `zrpe_cross_cohort_comparison.csv`, `zrpe_cross_cohort_forest.pdf` (harmonized formula, no age/sex).
